In [1]:
%matplotlib qt
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from operator import itemgetter
from PyPDF2 import PdfMerger

# plt.style.use('dark_background')

In [2]:
def mergePDFFiles(src_dir, tar_dir, tar_fname):
    '''
    Merges all PDF files in `src_dict` into one PDF file and stores it
    in `tar_dir` as `tar_fname`.
    '''
    # Find all PDF files
    pdfs = [os.path.join(src_dir, f) for f in os.listdir(src_dir) if f[-4:] == '.pdf']
    # Merge all found PDF files
    merger = PdfMerger()
    for pdf in pdfs:
        merger.append(pdf)
    # Save merged PDF file and close PdfMerger() object
    fpath_merged = os.path.join(tar_dir, tar_fname)
    merger.write(fpath_merged)
    merger.close()
    return None

In [3]:
# Given
spec_mode = 'fft'
compute_fmlt_mode = 'rest1'; compute_lt_rel_mode = 'rest1'
dir_rel = os.path.expanduser('~/research/results/wash-u/reliability/' + spec_mode)
dir_fmlt = os.path.expanduser('~/research/results/wash-u/familiality/' + spec_mode)
dir_results = os.path.expanduser('~/research/results/wash-u/')
fname_st_rel = 'shortterm_reliability_values.csv'
fname_lt_rel = 'longterm_reliability_values_compmode=' + compute_lt_rel_mode + '.csv'
fname_fmlt = 'familiality_values_compmode=' + compute_fmlt_mode + '.csv'
fpath_st_rel = os.path.join(dir_rel, fname_st_rel)
fpath_lt_rel = os.path.join(dir_rel, fname_lt_rel)
fpath_fmlt = os.path.join(dir_fmlt, fname_fmlt)
annot_size = 10

# Set plotting parameters
plt.rcParams['figure.figsize'] = [16, 7]
plt.rcParams['font.size'] = 14

# Create a colormap to have a fixed color to represent each feature
feat_names = ['F%d' % (i + 1) for i in range(50) if i not in range(3, 14)]
colors = cm.rainbow(np.linspace(0, 1, len(feat_names)))
cmap_dict = {}
for F, c in zip(feat_names, colors):
    cmap_dict[F] = c

# Reliability and MTCs distribution (ABSOLUTE VALUES!!) with features colored as per freq bands

In [33]:
# Create an alternate colormap with color representing freq band of each feature
band2feat_map = {
    'Delta': ['F16', 'F28'],
    'Theta': ['F17', 'F29'],
    'Alpha': ['F18', 'F19', 'F26', 'F30', 'F31', 'F38', 'F44', 'F48'],
    'Beta': ['F1', 'F2', 'F3', 'F20', 'F21', 'F22', 'F23', 'F24', 'F27',
             'F32', 'F33', 'F34', 'F35', 'F36', 'F39', 'F45', 'F49'],
    'Mixed': ['F15', 'F25', 'F37', 'F40', 'F41', 'F42', 'F43', 'F46', 'F47', 'F50'],
}   # mapping from freq bands to the features they contain
colors = cm.viridis_r(np.linspace(0, 1, len(band2feat_map)))
color_list = colors.tolist()[:-1] + ["none"]   # change color of 'Mixed' feats
# color_list = ['deeppink', 'darkorchid', 'mediumturquoise', 'darkblue', 'none']


# Read correlation values (e.g., short-term reliability)
corr_fpaths = [fpath_st_rel, fpath_lt_rel, fpath_fmlt]
corr_conds = ['Mean', 'Age-18Vs21', 'Mean']

plt.figure(figsize=(14, 7))    # create empty new figure
for i_corr, (corr_fpath, corr_cond) in enumerate(zip(corr_fpaths, corr_conds)):
    # corr_cond = 'Mean'

    # Create a dictionary of abs. corr values for corr_cond
    df_corr = pd.read_csv(corr_fpath)
    corr_dict = {}
    for i, v in enumerate(df_corr[corr_cond]):
        if not pd.isna(v):
            corr_dict['F%d' % (i + 1)] = np.abs(v)  # ABSOLUTE CORRELATION COEFFS

    # Plot and save corr values of all features
    for i_band, band in enumerate(band2feat_map):
        corr_band = [100 * corr_dict[Fs] for Fs in band2feat_map[band]] # corrs in %age for feats in given band
        if i_corr == 0:
            plt.scatter([i_corr + 1] * len(corr_band), corr_band, label=band,
                        edgecolors="k", color=color_list[i_band], alpha=0.8, s=100)
        else:
            plt.scatter([i_corr + 1] * len(corr_band), corr_band,
                        edgecolors="k", color=color_list[i_band], alpha=0.8, s=100)
plt.legend(loc="upper left")
plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
plt.xlim([0, 3.5])
plt.ylabel('Correlation Coefficient (%)')
plt.tight_layout()
fpath_fig = os.path.join(dir_results, 'demos', 'CollectionAbsCorrelations.pdf')
plt.savefig(fpath_fig, bbox_inches='tight')
plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()

# Sort short-term reliability values in descending order

In [ ]:
# Read short-term reliability values
df_st_rel = pd.read_csv(fpath_st_rel)

for st_rel_cond in df_st_rel.columns[1:]:
    # Create a dictionary of st-rel values for st_rel_cond (e.g. AgeGroup-4)
    st_rel_dict = {}
    for i, v in enumerate(df_st_rel[st_rel_cond]):
        if not pd.isna(v):
            st_rel_dict['F%d' % (i + 1)] = v

    # Sort st-rel values in the descending order
    st_rel_dict_sorted = dict(sorted(st_rel_dict.items(), key=itemgetter(1), reverse=True))

    # Plot and save sorted st-rel values of all features
    plt.figure()
    for i_Fs, Fs in enumerate(st_rel_dict_sorted):
        plt.scatter(i_Fs, st_rel_dict_sorted[Fs], color=cmap_dict[Fs])
        plt.annotate(Fs, (i_Fs, st_rel_dict_sorted[Fs]), size=annot_size)
    plt.axhline(y=0.5, color='r', ls='-')
    plt.ylim([-0.2, 1])
    plt.ylabel('Reliability')
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.title('%s: Short-term reliability (%s)' % (spec_mode.upper(), st_rel_cond))
    fpath_fig = os.path.join(dir_rel, 'shortterm_reliability_ordered_%s.pdf' % st_rel_cond)
    plt.savefig(fpath_fig, bbox_inches='tight')
    plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
    plt.show()

# Correlation between short-term reliabilities of features at ages 18 and 21

In [ ]:
# Read short-term reliability values
df_st_rel = pd.read_csv(fpath_st_rel)
st_rel_grp4 = [v for v in df_st_rel['Age-18'] if not pd.isna(v)]
st_rel_grp5 = [v for v in df_st_rel['Age-21'] if not pd.isna(v)]

# Evaluate correlation between st-rel values at ages 18 and 21
rho = np.corrcoef(st_rel_grp4, st_rel_grp5)
print('Correlation between short-term reliability values at ages 18 and 21 is %0.2f.' % rho[0, 1])

# Draw scatter plot of reliability values and save figure
plt.figure(figsize=(9, 6))
for i_F, F in enumerate(feat_names):
    plt.scatter(st_rel_grp4[i_F], st_rel_grp5[i_F], color=cmap_dict[F])
    plt.annotate(F, (st_rel_grp4[i_F], st_rel_grp5[i_F]), size=annot_size)
plt.ylim([-0.2, 1])
plt.title('%s: Corr Coef = %0.2f' % (spec_mode.upper(), rho[0, 1]))
plt.xlabel('Short-term reliability (Age-18)')
plt.ylabel('Short-term reliability (Age-21)')
fpath_fig = os.path.join(dir_rel, 'shortterm_reliability_comparison_Age-18Vs21.pdf')
plt.savefig(fpath_fig, bbox_inches='tight')
plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()

# Sort long-term reliability values in descending order

In [ ]:
# Read long-term reliability values
df_lt_rel = pd.read_csv(fpath_lt_rel)
lt_rel_cond = df_lt_rel.columns[1]

# Create a dictionary of lt-rel values for lt_rel_cond (i.e. Age-18Vs21)
lt_rel_dict = {}
for i, v in enumerate(df_lt_rel[lt_rel_cond]):
    if not pd.isna(v):
        lt_rel_dict['F%d' % (i + 1)] = v

# Sort lt-rel values in the descending order
lt_rel_dict_sorted = dict(sorted(lt_rel_dict.items(), key=itemgetter(1), reverse=True))

# Plot and save sorted lt-rel values of all features
plt.figure()
for i_Fs, Fs in enumerate(lt_rel_dict_sorted):
    plt.scatter(i_Fs, lt_rel_dict_sorted[Fs], color=cmap_dict[Fs])
    plt.annotate(Fs, (i_Fs, lt_rel_dict_sorted[Fs]), size=annot_size)
plt.axhline(y=0.5, color='r', ls='-')
plt.ylim([-0.2, 1])
plt.ylabel('Reliability')
plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
plt.title('%s: Long-term reliability (%s)' % (spec_mode.upper(), lt_rel_cond))
fpath_fig = os.path.join(dir_rel, 'longterm_reliability_ordered_compmode=%s_%s.pdf'
                         % (compute_lt_rel_mode, lt_rel_cond))
plt.savefig(fpath_fig, bbox_inches='tight')
plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()

# Combine all reliability results into an Appendix document

In [ ]:
mergePDFFiles(dir_rel, dir_results, 'Appendix-Reliability_SpecMode=%s.pdf' % spec_mode.upper())

# Sort familiality values in descending order

In [ ]:
# Read familiality values
df_fmlt = pd.read_csv(fpath_fmlt)

for fmlt_cond in df_fmlt.columns[1:]:
    # Create a dictionary of fmlt values for fmlt_cond (e.g. AgeGroup-4)
    fmlt_dict = {}
    for i, v in enumerate(df_fmlt[fmlt_cond]):
        if not pd.isna(v):
            fmlt_dict['F%d' % (i + 1)] = v

    # Sort fmlt values in the descending order
    fmlt_dict_sorted = dict(sorted(fmlt_dict.items(), key=itemgetter(1), reverse=True))

    # Plot and save sorted familiality values of all features
    plt.figure()
    for i_Fs, Fs in enumerate(fmlt_dict_sorted):
        plt.scatter(i_Fs, fmlt_dict_sorted[Fs], color=cmap_dict[Fs])
        plt.annotate(Fs, (i_Fs, fmlt_dict_sorted[Fs]), size=annot_size)
    plt.axhline(y=0.5, color='r', ls='-')
    plt.ylim([-0.2, 1])
    plt.ylabel('Familiality')
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.title('%s: Familiality (%s)' % (spec_mode.upper(), fmlt_cond))
    fpath_fig = os.path.join(dir_fmlt, 'familiality_ordered_compmode=%s_%s.pdf' %
                             (compute_fmlt_mode, fmlt_cond))
    plt.savefig(fpath_fig, bbox_inches='tight')
    plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
    plt.show()

# Correlation between familialities of features at ages 18 and 21

In [ ]:
# Read familiality values
df_fmlt = pd.read_csv(fpath_fmlt)
fmlt_grp4 = [v for v in df_fmlt['Age-18'] if not pd.isna(v)]
fmlt_grp5 = [v for v in df_fmlt['Age-21'] if not pd.isna(v)]

# Evaluate correlation between fmlt values at ages 18 and 21
rho = np.corrcoef(fmlt_grp4, fmlt_grp5)
print('Correlation between familiality values at ages 18 and 21 is %0.2f.' % rho[0, 1])

# Draw scatter plot of familiality values and save figure
plt.figure(figsize=(9, 6))
for i_F, F in enumerate(feat_names):
    plt.scatter(fmlt_grp4[i_F], fmlt_grp5[i_F], color=cmap_dict[F])
    plt.annotate(F, (fmlt_grp4[i_F], fmlt_grp5[i_F]), size=annot_size)
plt.ylim([-0.2, 1])
plt.title('%s: Corr Coef = %0.2f' % (spec_mode.upper(), rho[0, 1]))
plt.xlabel('Familiality (Age-18)')
plt.ylabel('Familiality (Age-21)')
fpath_fig = os.path.join(dir_fmlt, 'familiality_comparison_compmode=%s_Age-18Vs21.pdf' %
                         compute_fmlt_mode)
plt.savefig(fpath_fig, bbox_inches='tight')
plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()

# Combine all familiality results into an Appendix document

In [ ]:
mergePDFFiles(dir_fmlt, dir_results, 'Appendix-Familiality_SpecMode=%s.pdf' % spec_mode.upper())